# Oxy-fuel CCPP: design-point calculator

Runs the Section 3.2 equations of the model document (Akbari et al., Table S.2 plus the added closures) with real property data and returns the per-tonne coefficients for the PyPSA model.

**How to use**
1. Change values in the `params` cell (pressures, temperatures, efficiencies, λ, …).
2. Run all cells. The base case prints the state table, the plant performance and the PyPSA coefficients.
3. Use `sweep()` to see how results change with one parameter (examples for condenser pressure, turbine outlet pressure and turbine inlet temperature).
4. The last cell writes the coefficients to `ccpp_coefficients.csv`.

**Property data**
- Water/steam: CoolProp (`"Water"`, IAPWS formulation).
- Gas side (CO₂/O₂/H₂O): ideal-gas mixture. Each species' temperature-dependent heat capacity comes from CoolProp (`CP0MASS`); enthalpy and entropy tables are built from it once. Gas enthalpies are sensible enthalpies counted from 298.15 K, the LHV reference.

Requires: `pip install CoolProp numpy pandas scipy`

In [12]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from CoolProp.CoolProp import PropsSI

pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 60)

## 1. Inputs

Units: temperatures in K, pressures in bar, flows in kg/s, energies in kJ/kg. Results scale linearly with `m_bg`, so 1 kg/s is a convenient basis.

In [13]:
params = {
    # Biogas and fuel
    "m_bg": 1.0,          # kg/s biogas (basis)
    "y_CH4": 0.60,        # CH4 mole fraction of biogas (Akbari 2.3.1), rest CO2
    "LHV_CH4": 50_000.0,  # kJ/kg
    "lam": 1.05,          # oxygen excess ratio (choice)

    # Compressor and combustion chamber
    "T3": 298.15,         # K, oxidant at compressor inlet (assumed)
    "P3": 1.013,          # bar, compressor inlet (assumed)
    "PR": 14.5,            # pressure ratio (Akbari Table 8)
    "eta_comp": 0.85,     # isentropic efficiency (Akbari 2.3.2)
    "T5": 1261.0,         # K, turbine inlet temperature (Akbari Table 8)
    "eta_cch": 0.99,      # combustion efficiency (Akbari 2.3.2)
    "dp_cch": 0.03,       # relative pressure loss (Akbari 2.3.2)

    # Gas turbine
    "eta_gt": 0.88,       # isentropic efficiency (Akbari 2.3.2)
    "P6": 1.05,           # bar, turbine outlet (assumed)

    # Steam cycle
    "P_up": 133.0,        # bar, live steam / HRSG pressure (13.3 MPa, Akbari Table 8)
    "dT_sup": 108.8,      # K, superheat above saturation (Akbari Table 8)
    "dT_pinch": 30.0,     # K, HRSG pinch (Akbari 2.3.2)
    "P_cond": 0.05,       # bar, condenser pressure (assumed)
    "eta_st": 0.85,       # steam turbine isentropic efficiency (Akbari 2.3.2)
    "eta_fp": 0.80,       # feed pump isentropic efficiency (Akbari 2.3.2)
    "eta_gen": 0.96,      # electrical conversion efficiency (Akbari 2.3.2)

    # Downstream of the HRSG
    "T10": 333.15,        # K, flue gas after the digester heat exchanger (assumed, 60 °C)
}

## 2. Property functions

Gas side: ideal-gas tables per species (1 K steps, 280–2000 K) built from CoolProp heat capacities. Mixture enthalpy and entropy follow from the mole fractions. The entropy mixing term is left out because the composition does not change inside a compressor or turbine, so it cancels.

Water side: thin wrapper around CoolProp, with pressures in bar and enthalpies in kJ/kg.

In [14]:
R = 8.314462618                                  # J/(mol K)
SPECIES = ["CO2", "O2", "H2O"]
M = {s: PropsSI("M", s) for s in SPECIES}        # kg/mol
M_CH4 = PropsSI("M", "Methane")
T_REF = 298.15
T_GRID = np.linspace(280.0, 2000.0, 1721)


def _ideal_gas_table(species):
    # cp in J/(mol K); h and s integrated from 298.15 K
    cp = np.array([PropsSI("CP0MASS", "T", T, "P", 100.0, species) for T in T_GRID]) * M[species]
    dT = np.diff(T_GRID)
    h = np.concatenate([[0.0], np.cumsum(0.5 * (cp[1:] + cp[:-1]) * dT)])
    s = np.concatenate([[0.0], np.cumsum(0.5 * (cp[1:] / T_GRID[1:] + cp[:-1] / T_GRID[:-1]) * dT)])
    return h - np.interp(T_REF, T_GRID, h), s - np.interp(T_REF, T_GRID, s)


TABLE = {s: _ideal_gas_table(s) for s in SPECIES}


def mole_fractions(n):
    total = sum(n.values())
    return {k: v / total for k, v in n.items() if v > 0}


def molar_mass(y):
    return sum(y[s] * M[s] for s in y)


def gas_h(T, y):
    # kJ/kg, sensible enthalpy relative to 298.15 K
    return sum(y[s] * np.interp(T, T_GRID, TABLE[s][0]) for s in y) / molar_mass(y) / 1000


def gas_s(T, P, y):
    # kJ/(kg K), relative value (enough for isentropic calculations)
    s_mol = sum(y[s] * np.interp(T, T_GRID, TABLE[s][1]) for s in y) - R * np.log(P / 1.01325)
    return s_mol / molar_mass(y) / 1000


def gas_T_from_h(h, y):
    return brentq(lambda T: gas_h(T, y) - h, T_GRID[0], T_GRID[-1])


def gas_T_from_s(s, P, y):
    return brentq(lambda T: gas_s(T, P, y) - s, T_GRID[0], T_GRID[-1])


def water(out, name1, value1, name2, value2):
    # CoolProp for water; P in bar, h in kJ/kg, s in kJ/(kg K), T in K
    scale_in = {"P": 1e5, "H": 1e3, "S": 1e3, "T": 1.0, "Q": 1.0}
    scale_out = {"P": 1e5, "H": 1e3, "S": 1e3, "T": 1.0, "Q": 1.0, "D": 1.0}
    v = PropsSI(out, name1, value1 * scale_in[name1], name2, value2 * scale_in[name2], "Water")
    return v / scale_out[out]

## 3. The CCPP model

One function, in the solving order of Section 3.2:

1. **Combustion chamber + compressor** (iterated): the oxidant composition depends on the oxidant flow, and the oxidant flow follows from the energy balance at fixed T₅.
2. **Gas turbine**: isentropic expansion to P₆, corrected with η_GT.
3. **Feed pump and water states** from steam tables.
4. **HRSG**: the minimum gas–water temperature gap (`dT_pinch`) must hold at the evaporator cold end (state 8) and at the economiser cold end (state 9). Each gives a maximum steam flow; the smaller one is used, and the result reports which end limits. The section balances then give T₇, T₈ and T₉.
5. **Steam turbine and condenser**.
6. **Downstream**: heat to the digester, water knock-out, CO₂ split.
7. **Net output, efficiency and PyPSA coefficients** per tonne of biogas.

Warnings are collected when a check fails (HRSG hot end too close, wet steam turbine exhaust).

In [15]:
def run_ccpp(p):
    m_bg = p["m_bg"]
    warnings = []

    # --- Biogas and stoichiometry (CH4 + 2 O2 -> CO2 + 2 H2O) ---
    x_CH4 = p["y_CH4"] * M_CH4 / (p["y_CH4"] * M_CH4 + (1 - p["y_CH4"]) * M["CO2"])
    n_CH4 = x_CH4 * m_bg / M_CH4                       # mol/s
    n_CO2_bg = (1 - x_CH4) * m_bg / M["CO2"]           # mol/s
    n_O2 = 2 * p["lam"] * n_CH4                        # mol/s, from the PEM
    Q_fuel = x_CH4 * m_bg * p["LHV_CH4"]               # kW
    P4 = p["PR"] * p["P3"]
    P5 = (1 - p["dp_cch"]) * P4

    def chamber(m_ox):
        n_rec = (m_ox - n_O2 * M["O2"]) / M["CO2"]
        y_ox = mole_fractions({"O2": n_O2, "CO2": n_rec})
        # Compressor 3 -> 4
        h3 = gas_h(p["T3"], y_ox)
        T4s = gas_T_from_s(gas_s(p["T3"], p["P3"], y_ox), P4, y_ox)
        h4 = h3 + (gas_h(T4s, y_ox) - h3) / p["eta_comp"]
        # Products at state 5
        n_g = {"CO2": n_CO2_bg + n_CH4 + n_rec, "H2O": 2 * n_CH4, "O2": 2 * (p["lam"] - 1) * n_CH4}
        y_g = mole_fractions(n_g)
        h5 = gas_h(p["T5"], y_g)
        # Energy balance solved for the oxidant flow
        m_ox_new = (x_CH4 * m_bg * p["LHV_CH4"] * p["eta_cch"] - m_bg * h5) / (h5 - h4)
        return dict(n_rec=n_rec, y_ox=y_ox, h3=h3, h4=h4, n_g=n_g, y_g=y_g, h5=h5, m_ox_new=m_ox_new)

    # --- Combustion chamber + compressor, fixed-point iteration on the oxidant flow ---
    m_ox = 15.0 * m_bg
    for _ in range(100):
        c = chamber(m_ox)
        converged = abs(c["m_ox_new"] - m_ox) < 1e-9 * m_bg
        m_ox = c["m_ox_new"]
        if converged:
            break
    c = chamber(m_ox)
    y_ox, y_g, n_g, n_rec = c["y_ox"], c["y_g"], c["n_g"], c["n_rec"]
    h3, h4, h5 = c["h3"], c["h4"], c["h5"]
    T4 = gas_T_from_h(h4, y_ox)
    m_g = m_bg + m_ox
    assert abs(sum(n_g[k] * M[k] for k in n_g) - m_g) < 1e-6 * m_g, "mass balance does not close"
    W_comp = m_ox * (h4 - h3)
    y_O2_ox = y_ox["O2"]

    # --- Gas turbine 5 -> 6 ---
    T6s = gas_T_from_s(gas_s(p["T5"], P5, y_g), p["P6"], y_g)
    h6 = h5 - p["eta_gt"] * (h5 - gas_h(T6s, y_g))
    T6 = gas_T_from_h(h6, y_g)
    W_gt = m_g * (h5 - h6)

    # --- Water states and feed pump 20 -> 21 ---
    Pup, Pc = p["P_up"], p["P_cond"]
    T_sat = water("T", "P", Pup, "Q", 0)
    h22, h23 = water("H", "P", Pup, "Q", 0), water("H", "P", Pup, "Q", 1)
    T24 = T_sat + p["dT_sup"]
    h24, s24 = water("H", "P", Pup, "T", T24), water("S", "P", Pup, "T", T24)
    T20 = water("T", "P", Pc, "Q", 0)
    h20, s20 = water("H", "P", Pc, "Q", 0), water("S", "P", Pc, "Q", 0)
    h21 = h20 + (water("H", "P", Pup, "S", s20) - h20) / p["eta_fp"]
    T21 = water("T", "P", Pup, "H", h21)

    # --- HRSG 6 -> 9 (gas), 21 -> 24 (water) ---
    # The minimum gas-water temperature gap (dT_pinch) must hold at the evaporator
    # cold end (state 8) AND at the economiser cold end (state 9). Each gives a
    # maximum steam flow; the smaller one limits the HRSG.
    if T6 - p["dT_pinch"] <= T_sat:
        raise ValueError(f"Turbine exhaust ({T6 - 273.15:.0f} °C) is too cold to raise steam at {Pup} bar")
    m_rc_evap = m_g * (h6 - gas_h(T_sat + p["dT_pinch"], y_g)) / (h24 - h22)
    m_rc_cold = m_g * (h6 - gas_h(T21 + p["dT_pinch"], y_g)) / (h24 - h21)
    m_rc = min(m_rc_evap, m_rc_cold)
    hrsg_limit = "evaporator pinch" if m_rc_evap <= m_rc_cold else "economiser cold end"
    h7 = h6 - m_rc * (h24 - h23) / m_g
    h8 = h7 - m_rc * (h23 - h22) / m_g
    h9 = h8 - m_rc * (h22 - h21) / m_g
    T7, T8, T9 = gas_T_from_h(h7, y_g), gas_T_from_h(h8, y_g), gas_T_from_h(h9, y_g)
    Q_hrsg = m_g * (h6 - h9)
    if T6 - T24 < p["dT_pinch"]:
        warnings.append(f"HRSG hot end: gas only {T6 - T24:.0f} K above the live steam")

    # --- Steam turbine 24 -> 25 and condenser 25 -> 20 ---
    h25 = h24 - p["eta_st"] * (h24 - water("H", "P", Pc, "S", s24))
    x25 = water("Q", "P", Pc, "H", h25)
    if not 0 <= x25 <= 1:
        x25 = 1.0  # superheated exhaust
    W_st = m_rc * (h24 - h25)
    W_fp = m_rc * (h21 - h20)
    Q_cond = m_rc * (h25 - h20)
    if x25 < 0.90:
        warnings.append(f"Steam turbine exhaust is {x25:.0%} vapour (usual minimum about 90 %)")

    # --- Downstream of the HRSG ---
    h10 = gas_h(p["T10"], y_g)
    Q_ad = max(m_g * (h9 - h10), 0.0)        # sensible heat only, condensation not counted
    m_H2O = n_g["H2O"] * M["H2O"]
    m_CO2_new = (n_CO2_bg + n_CH4) * M["CO2"]
    m_CO2_rec = n_rec * M["CO2"]
    m_O2 = n_O2 * M["O2"]

    # --- Net output and efficiency ---
    W_net = p["eta_gen"] * (W_gt + W_st) - W_comp - W_fp
    eta = W_net / Q_fuel

    C = lambda T: T - 273.15
    states = pd.DataFrame(
        [
            ("3", "Oxidant, compressor inlet", C(p["T3"]), p["P3"], h3, m_ox, None),
            ("4", "Oxidant, compressor outlet", C(T4), P4, h4, m_ox, None),
            ("5", "Hot gas, turbine inlet", C(p["T5"]), P5, h5, m_g, None),
            ("6", "Turbine exhaust", C(T6), p["P6"], h6, m_g, None),
            ("7", "After superheater", C(T7), p["P6"], h7, m_g, None),
            ("8", "After evaporator (pinch)", C(T8), p["P6"], h8, m_g, None),
            ("9", "HRSG exit", C(T9), p["P6"], h9, m_g, None),
            ("10", "After digester heat exchanger", C(p["T10"]), p["P6"], h10, m_g, None),
            ("20", "Condensate", C(T20), Pc, h20, m_rc, 0.0),
            ("21", "Feedwater after pump", C(T21), Pup, h21, m_rc, None),
            ("22", "Saturated liquid", C(T_sat), Pup, h22, m_rc, 0.0),
            ("23", "Saturated vapour", C(T_sat), Pup, h23, m_rc, 1.0),
            ("24", "Superheated steam", C(T24), Pup, h24, m_rc, None),
            ("25", "Steam turbine exhaust", C(T20), Pc, h25, m_rc, x25),
        ],
        columns=["state", "description", "T [°C]", "P [bar]", "h [kJ/kg]", "m [kg/s]", "vapour fraction"],
    ).set_index("state").round(3)

    performance = pd.Series({
        "fuel input (LHV) [MW]": Q_fuel / 1e3,
        "oxidant flow [kg/s]": m_ox,
        "O2 mole fraction of oxidant [-]": y_O2_ox,
        "hot gas flow [kg/s]": m_g,
        "steam flow [kg/s]": m_rc,
        "HRSG limited by": hrsg_limit,
        "compressor [MW]": W_comp / 1e3,
        "gas turbine [MW]": W_gt / 1e3,
        "steam turbine [MW]": W_st / 1e3,
        "feed pump [MW]": W_fp / 1e3,
        "net electricity [MW]": W_net / 1e3,
        "eta_CCPP [-]": eta,
        "HRSG duty [MW]": Q_hrsg / 1e3,
        "condenser heat [MW]": Q_cond / 1e3,
        "heat to digester [MW]": Q_ad / 1e3,
        "HRSG balance error [kW]": Q_hrsg - m_rc * (h24 - h21),
        # fuel + compressor + pump + oxidant enthalpy in = turbines + condenser + exit gas + combustion loss out
        "plant energy balance error [kW]": (Q_fuel + W_comp + W_fp + m_ox * h3)
        - (W_gt + W_st + Q_cond + m_g * h9 + (1 - p["eta_cch"]) * Q_fuel),
    })

    per_t = lambda kW: kW / m_bg / 3600   # kW per (kg/s biogas) -> MWh per t biogas
    coefficients = pd.Series({
        "fuel (LHV) [MWh/t biogas]": per_t(Q_fuel),
        "electricity [MWh/t biogas]": per_t(W_net),
        "eta_CCPP [-]": eta,
        "O2 from PEM [t/t biogas]": m_O2 / m_bg,
        "new CO2 [t/t biogas]": m_CO2_new / m_bg,
        "water [t/t biogas]": m_H2O / m_bg,
        "r_ox, recycled CO2 per O2 [t/t]": m_CO2_rec / m_O2,
        "heat to digester [MWh/t biogas]": per_t(Q_ad),
        "condenser heat [MWh/t biogas]": per_t(Q_cond),
    })

    return dict(states=states, performance=performance, coefficients=coefficients, warnings=warnings)

## 4. Base case

In [16]:
res = run_ccpp(params)
display(res["states"])

,description,T [°C],P [bar],h [kJ/kg],m [kg/s],vapour fraction
state,,,,,,
3,"Oxidant, compressor inlet",25.000,1.013,0.000,18.557,NaN
4,"Oxidant, compressor outlet",282.668,14.688,246.285,18.557,NaN
5,"Hot gas, turbine inlet",987.850,14.248,1128.441,19.557,NaN
6,Turbine exhaust,619.140,1.050,650.920,19.557,NaN
7,After superheater,541.371,1.050,555.379,19.557,NaN
8,After evaporator (pinch),362.634,1.050,345.294,19.557,NaN
9,HRSG exit,114.325,1.050,82.897,19.557,NaN
10,After digester heat exchanger,60.000,1.050,31.600,19.557,NaN
20,Condensate,32.874,0.050,137.749,3.695,0.000


In [17]:
display(res["performance"].to_frame("value"))
display(res["coefficients"].to_frame("value"))
for w in res["warnings"]:
    print("Warning:", w)

,value
fuel input (LHV) [MW],17.675
oxidant flow [kg/s],18.557
O2 mole fraction of oxidant [-],0.107
hot gas flow [kg/s],19.557
steam flow [kg/s],3.695
HRSG limited by,evaporator pinch
compressor [MW],4.57
gas turbine [MW],9.339
steam turbine [MW],3.999
feed pump [MW],0.062


,value
fuel (LHV) [MWh/t biogas],4.910
electricity [MWh/t biogas],2.270
eta_CCPP [-],0.462
O2 from PEM [t/t biogas],1.481
new CO2 [t/t biogas],1.616
water [t/t biogas],0.794
"r_ox, recycled CO2 per O2 [t/t]",11.532
heat to digester [MWh/t biogas],0.279
condenser heat [MWh/t biogas],1.992


Gas enthalpies are sensible enthalpies counted from 298.15 K; water enthalpies use the steam-table reference (liquid at 0.01 °C). Only differences within each fluid are used, so the two references never mix.

## 5. Sweeps

`sweep(name, values)` reruns the model for each value of one parameter and returns a table of the main results.

In [18]:
def sweep(name, values, base=params):
    rows = []
    for v in values:
        try:
            r = run_ccpp({**base, name: v})
        except ValueError as e:
            rows.append({name: v, "note": str(e)})
            continue
        perf, coef = r["performance"], r["coefficients"]
        rows.append({
            name: v,
            "eta_CCPP": perf["eta_CCPP [-]"],
            "net el. [MW]": perf["net electricity [MW]"],
            "T6 [°C]": r["states"].loc["6", "T [°C]"],
            "T9 [°C]": r["states"].loc["9", "T [°C]"],
            "steam [kg/s]": perf["steam flow [kg/s]"],
            "HRSG limit": perf["HRSG limited by"],
            "exhaust vapour": r["states"].loc["25", "vapour fraction"],
            "condenser heat [MW]": perf["condenser heat [MW]"],
            "r_ox [t/t]": coef["r_ox, recycled CO2 per O2 [t/t]"],
            "note": "; ".join(r["warnings"]),
        })
    return pd.DataFrame(rows).set_index(name)

In [19]:
# Condenser pressure: pure power (low) vs combined heat and power (high)
sweep("P_cond", [0.03, 0.05, 0.1, 0.2, 0.5, 1.0])

,eta_CCPP,net el. [MW],T6 [°C],T9 [°C],steam [kg/s],HRSG limit,exhaust vapour,condenser heat [MW],r_ox [t/t],note
P_cond,,,,,,,,,,
0.03,0.471,8.326,619.14,107.118,3.695,evaporator pinch,0.792,7.147,11.532,Steam turbine exhaust is 79% vapour (usual minimum about...
0.05,0.462,8.173,619.14,114.325,3.695,evaporator pinch,0.801,7.171,11.532,Steam turbine exhaust is 80% vapour (usual minimum about...
0.10,0.450,7.953,619.14,124.843,3.695,evaporator pinch,0.815,7.200,11.532,Steam turbine exhaust is 81% vapour (usual minimum about...
0.20,0.437,7.718,619.14,136.339,3.695,evaporator pinch,0.829,7.224,11.532,Steam turbine exhaust is 83% vapour (usual minimum about...
0.50,0.418,7.382,619.14,153.332,3.695,evaporator pinch,0.851,7.244,11.532,Steam turbine exhaust is 85% vapour (usual minimum about...
1.00,0.402,7.106,619.14,167.830,3.695,evaporator pinch,0.869,7.247,11.532,Steam turbine exhaust is 87% vapour (usual minimum about...


In [20]:
# Gas turbine outlet pressure (back-pressure from the HRSG and downstream equipment)
sweep("P6", [1.013, 1.05, 1.10, 1.20])

,eta_CCPP,net el. [MW],T6 [°C],T9 [°C],steam [kg/s],HRSG limit,exhaust vapour,condenser heat [MW],r_ox [t/t],note
P6,,,,,,,,,,
1.013,0.464,8.207,614.879,119.009,3.631,evaporator pinch,0.801,7.047,11.532,Steam turbine exhaust is 80% vapour (usual minimum about...
1.050,0.462,8.173,619.140,114.325,3.695,evaporator pinch,0.801,7.171,11.532,Steam turbine exhaust is 80% vapour (usual minimum about...
1.100,0.460,8.128,624.696,108.181,3.778,evaporator pinch,0.801,7.333,11.532,Steam turbine exhaust is 80% vapour (usual minimum about...
1.200,0.455,8.045,635.178,96.471,3.936,evaporator pinch,0.801,7.639,11.532,Steam turbine exhaust is 80% vapour (usual minimum about...


In [21]:
# Turbine inlet temperature (Akbari Table 7 range: 1150-1650 K)
sweep("T5", [1150, 1261, 1400, 1550, 1650])

,eta_CCPP,net el. [MW],T6 [°C],T9 [°C],steam [kg/s],HRSG limit,exhaust vapour,condenser heat [MW],r_ox [t/t],note
T5,,,,,,,,,,
1150,0.409,7.233,535.881,201.792,2.953,evaporator pinch,0.801,5.731,14.274,Steam turbine exhaust is 80% vapour (usual minimum about...
1261,0.462,8.173,619.140,114.325,3.695,evaporator pinch,0.801,7.171,11.532,Steam turbine exhaust is 80% vapour (usual minimum about...
1400,0.489,8.638,723.183,64.005,4.004,economiser cold end,0.801,7.772,9.131,Steam turbine exhaust is 80% vapour (usual minimum about...
1550,0.495,8.747,835.210,64.005,4.006,economiser cold end,0.801,7.776,7.307,Steam turbine exhaust is 80% vapour (usual minimum about...
1650,0.498,8.801,909.766,64.005,4.009,economiser cold end,0.801,7.780,6.376,Steam turbine exhaust is 80% vapour (usual minimum about...


## 6. Export the coefficients for PyPSA

In [ ]:
res["coefficients"].to_csv("ccpp_coefficients.csv", header=["value"])
res["coefficients"]

fuel (LHV) [MWh/t biogas]           4.910
electricity [MWh/t biogas]          2.270
eta_CCPP [-]                        0.462
O2 from PEM [t/t biogas]            1.481
new CO2 [t/t biogas]                1.616
water [t/t biogas]                  0.794
r_ox, recycled CO2 per O2 [t/t]    11.532
heat to digester [MWh/t biogas]     0.279
condenser heat [MWh/t biogas]       1.992
dtype: float64

: 